# Fake News Classification: DistilBERT vs. Logistic Regression
### Empirical Evaluation, Error Analysis, and Epistemic Limits of Text Classification

This notebook demonstrates:
1. **Data Loading & Stratification**: Partitioning the benchmark dataset.
2. **Baseline Model**: Sublinear TF-IDF + L2 Logistic Regression.
3. **Transformer Fine-Tuning**: Fine-tuned DistilBERT on Apple Silicon (MPS).
4. **Comparative Analysis**: Metrics, confusion matrices, and ROC curves.
5. **Error Breakdown**: False Positives (Disinformation) and False Negatives (Censored Real News).
6. **Adversarial Probes**: Empirical proof that style inverts model truth perception.
7. **Epistemic Discussion**: Why text classification cannot independently verify factual truth.

In [ ]:
import os
import json
import torch
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ['HF_HUB_OFFLINE'] = '1'
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Execution Device: {device.type.upper()}')

## 1. Data Preparation & Splitting

In [ ]:
from data_loader import prepare_data

train_df, val_df, test_df = prepare_data(train_size=5000, val_size=1000, test_size=2000, seed=42)
print('\nSample Record:')
print(f"Label: {train_df.iloc[0]['label']} (0=Fake, 1=Real)")
print(f"Title: {train_df.iloc[0]['title']}")
print(f"Lead:  {train_df.iloc[0]['text'][:160]}...")

## 2. Baseline Model: TF-IDF + Logistic Regression

In [ ]:
from train_baseline import train_and_eval_logistic_regression

lr_clf, tfidf_vec, lr_results = train_and_eval_logistic_regression(train_df, val_df, test_df)

print('\nTop 10 Indicative Words for REAL news (positive weights):')
for word, coef in lr_results['top_real_features'][:10]:
    print(f'  {word:<22} : {coef:+.3f}')

print('\nTop 10 Indicative Words for FAKE news (negative weights):')
for word, coef in lr_results['top_fake_features'][:10]:
    print(f'  {word:<22} : {coef:+.3f}')

## 3. Pretrained & Fine-Tuned DistilBERT Evaluation

In [ ]:
# Load trained checkpoint
distilbert_path = './models/distilbert'
tokenizer = AutoTokenizer.from_pretrained(distilbert_path)
db_model = AutoModelForSequenceClassification.from_pretrained(distilbert_path)
db_model.to(device)

with open('./results/consolidated_metrics.json') as f:
    metrics = json.load(f)

db_m = metrics['distilbert']
print(f'DistilBERT Test Accuracy : {db_m["accuracy"]*100:.2f}%')
print(f'DistilBERT Macro F1      : {db_m["macro_f1"]:.4f}')
print(f'DistilBERT ROC-AUC       : {db_m["roc_auc"]:.4f}')

## 4. Visual Comparison: Confusion Matrices & Top Features

In [ ]:
display(Image(filename='./results/confusion_matrices.png'))
display(Image(filename='./results/top_features.png'))

## 5. Adversarial Probes: Style vs. Ground Truth

We test 5 hypothesis-driven probe articles across both models to demonstrate how linguistic style inverts truth perception.

In [ ]:
from adversarial_probes import run_probes

probe_results = run_probes(lr_clf, tfidf_vec, db_model, tokenizer, device)
display(Image(filename='./results/adversarial_probes.png'))

## 6. Epistemic Limitation: Why Classification Does Not Independently Verify Factual Truth

### The Epistemic Gap
A sequence classifier computes conditional probability:
$$P(Y = 1 \mid X = x_1, x_2, \dots, x_n)$$

This measures how closely the sequence matches the statistical genre conventions of the training set. However, according to the correspondence theory of truth, an assertion $S$ is true if and only if **the external world is in the state described by $S$**:
$$\text{True}(S) \iff \text{World} \models S$$

A statistical model has no connection to extrinsic world states. As our adversarial probes proved:
- An absurd falsehood (*'The Moon is made of cheese'*) written in Reuters wire style is classified as **REAL with 99.9% confidence**.
- A verified historical event (*2013 Chelyabinsk meteor explosion*) written with emotional tabloid markers is classified as **FAKE with 99.7% confidence**.

### The Architecture of Verifiable Fact-Checking
Real automated verification requires:
1. **Claim Extraction & Decomposition**
2. **Dynamic Evidence Retrieval** from authoritative knowledge bases
3. **Natural Language Inference (NLI)** (Entailment / Contradiction)
4. **Epistemic Provenance & Source Reliability Ranking**